# Viz Designer Selection Pipeline

Interactive Notebook for testing Viz Designer's chart selection step from simulated user query and data shape

In [1]:
%load_ext autoreload
%autoreload 2

### Setup API Keys

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")


### LangSmith configured before xlake or agent

In [3]:
# IMPORTANT: Set LangSmith environment variables BEFORE any LangChain imports
# The LangSmith client is initialized on first import and caches the endpoint
import os
import getpass

# Set up LangSmith for EU endpoint FIRST (before any other imports)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "viz_designer"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key (press Enter to skip): ") or ""

if not os.environ["LANGCHAIN_API_KEY"]:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled (no API key)")
else:
    print(f"LangSmith tracing enabled for EU endpoint")
    print(f"  Endpoint: {os.environ['LANGCHAIN_ENDPOINT']}")
    print(f"  Project: {os.environ['LANGCHAIN_PROJECT']}")


LangSmith tracing enabled for EU endpoint
  Endpoint: https://eu.api.smith.langchain.com
  Project: viz_designer


### Set Up XLake and Load Data Viz bible into RAG

In [4]:
# Create in-memory XLakeClient with all stores configured for testing
from pathlib import Path

from lib.xlake_utils import create_xlake_test_client

xlake_client = create_xlake_test_client()
print("XLakeClient initialized with all in-memory stores")

# ----------------------------------------------------------------
# Ingest data-viz-bible into the client's CoreContextStore
# ----------------------------------------------------------------
from xlake.utils import ingest_viz_bible

VIZ_BIBLE_PATH = Path("../../../knowledge/data-viz-bible")

# Pass the client's core_context store so data persists in the client
stats = ingest_viz_bible(
    VIZ_BIBLE_PATH,
    store=xlake_client.stores.core_context,
    verbose=True,
)

print(f"\nIngestion complete:")
print(f"  Files processed: {stats['files_processed']}")
print(f"  Chunks stored: {stats['chunks_stored']}")


XLakeClient initialized with all in-memory stores
Found 48 markdown files to process
Processing: 00-index.md
Processing: 01-schema-reference.md
Processing: 02-classification-system.md
Processing: 03-agent-ui-handoff.md
Processing: examples/example-01-sales-dashboard.md
Processing: examples/example-02-product-comparison.md
Processing: examples/example-03-financial-bridge.md
Processing: examples/example-04-trend-with-insight.md
Processing: refinement/refinement-area-chart.md
Processing: refinement/refinement-bar-chart-grouped.md
Processing: refinement/refinement-bar-chart-horizontal.md
Processing: refinement/refinement-bar-chart-stacked.md
Processing: refinement/refinement-bar-chart-vertical.md
Processing: refinement/refinement-boxplot.md
Processing: refinement/refinement-data-table.md
Processing: refinement/refinement-donut-chart.md
Processing: refinement/refinement-heatmap.md
Processing: refinement/refinement-histogram.md
Processing: refinement/refinement-kpi-card.md
Processing: refine

### Load the tools

In [5]:

# ----------------------------------------------------------------
# Create SearchTools and viz_rule_search_tool
# ----------------------------------------------------------------
from agents.tools.search_tools import SearchTools, make_viz_rules_search_tool

# Create SearchTools with the XLakeClient
search_tools = SearchTools(xlake_client)

# Create the viz rules search tool for loading into agents
viz_rule_search_tool = make_viz_rules_search_tool(search_tools)

print(f"\nSearchTools initialized")
print(f"  viz_rule_search_tool: {viz_rule_search_tool.name} description: {viz_rule_search_tool.description}")



SearchTools initialized
  viz_rule_search_tool: search_viz_design_rules description: Search for visualization design rules by semantic query.

        Searches the Data Viz Bible knowledge base for relevant rules about
        chart selection, refinement, formatting, and implementation.

        Args:
            query: Natural language query to search for relevant rules.
            runtime: ToolRuntime providing AgentContext with tenant/user.
            pipeline_stage: Filter by stage (selection|refinement|formatting|implementation).
            chart_types: Filter by chart types (e.g., ["line_chart", "bar_chart_vertical"]).
            top_k: Maximum number of results to return.

        Returns:
            List of VizDesignRuleResponse containing rule and retrieval stats.


### Create the agent from the config

In [6]:
# Create the viz selection agent using LangGraph's create_agent
from agents.viz_designer.agent import VisualizationDesigner
from agents.viz_designer.config import VisualizationDesignerSettings
from lib.agent_utils import build_selection_stage

# Create the selector agent with viz rules search tool
# Note: xlake_client is passed to VisualizationDesigner which automatically adds the search tool
settings = VisualizationDesignerSettings(
    default_model="gpt-4o-mini",
    debug=True,
)
viz_agent = VisualizationDesigner(settings=settings, xlake_client=xlake_client)
selector_agent = build_selection_stage(viz_agent)

print("Selector agent created with viz_rule_search_tool")

Selector agent created with viz_rule_search_tool


## Load Test Data and Prepare First Test Case

In [7]:
# Load test cases from YAML
from typing import Any

import yaml

with open("../../data_samples/nlp_chart_test_data.yaml") as f:
    test_data = yaml.safe_load(f)

test_cases: list[dict[str, Any]] = test_data["test_cases"]
print(f"Loaded {len(test_cases)} test cases:")
for tc in test_cases:
    expected = tc["expected_chart"]["chart_type"]
    print(f"  - {tc['id']}: expected chart_type = '{expected}'")

# Pick the first test case
first_test_case = test_cases[0]
print(f"\nUsing first test case: {first_test_case['id']}")
print(f"  NLP Query: {first_test_case['nlp_query']}")
print(f"  Expected Chart: {first_test_case['expected_chart']['chart_type']}")

Loaded 5 test cases:
  - tc_001_monthly_sales: expected chart_type = 'bar_chart_vertical'
  - tc_002_sales_by_region_product: expected chart_type = 'heatmap'
  - tc_003_customer_cohort_retention: expected chart_type = 'cohort_heatmap'
  - tc_004_revenue_vs_cost_trend: expected chart_type = 'combo'
  - tc_005_distribution_analysis: expected chart_type = 'boxplot'

Using first test case: tc_001_monthly_sales
  NLP Query: Show me the monthly sales numbers for the last twelve months
  Expected Chart: bar_chart_vertical


## Set Up Agent State and Context

In [8]:
# Set up agent context with pipeline_stage for selection
from agents.models import AgentContext
from xlake.models import TenantContext, TenantIdentity, UserContext

# Create tenant and user contexts (required for tool runtime)
tenant = TenantContext(
    identity=TenantIdentity(
        tenant_id="actbi",
        tenant_name="ActBI Test",
        industry="technology",
        region="US",
        timezone="America/New_York",
        locale="en_US",
    )
)
user = UserContext(user_id="notebook_user", tenant_id="actbi", role="admin")

# Create agent context
agent_context = AgentContext(tenant=tenant, user=user)

# Define the agent state with pipeline_stage="selection"
# This will be used when invoking the agent to filter search results
agent_state = {
    "pipeline_stage": "selection",
    "context": agent_context,
}

print(f"Agent state configured:")
print(f"  pipeline_stage: {agent_state['pipeline_stage']}")
print(f"  tenant: {agent_context.tenant.identity.tenant_id}")
print(f"  user: {agent_context.user.user_id}")

Agent state configured:
  pipeline_stage: selection
  tenant: actbi
  user: notebook_user


## Run the Agent

In [10]:
# Invoke the selector agent with VizSelectionState
# The state includes: nlp_query, output_schema, materialized_data
# The agent will use the viz rules search tool to find relevant chart selection criteria
from langchain_core.messages import HumanMessage

from agents.viz_designer.prompts import format_selection_request
from agents.viz_designer.schemas import VizSelectionState

# Format the request using the helper function
formatted_request = format_selection_request(
    nlp_query=first_test_case["nlp_query"],
    output_schema=first_test_case["output_schema"],
    materialized_data=first_test_case["materialized_data"]["rows"],
)

# Build the selection state from the first test case
selection_state: VizSelectionState = {
    "messages": [HumanMessage(content=formatted_request)],
    "nlp_query": first_test_case["nlp_query"],
    "output_schema": first_test_case["output_schema"],
    "materialized_data": first_test_case["materialized_data"]["rows"],
}

print("VizSelectionState prepared:")
print(f"  nlp_query: {selection_state['nlp_query']}")
print(f"  output_schema fields: {len(selection_state['output_schema']['fields'])}")
print(f"  materialized_data rows: {len(selection_state['materialized_data'])}")

# Invoke the agent with the selection state
# Note: context is passed as a keyword argument, not inside the config dict
result = await selector_agent.ainvoke(
    selection_state,
    {"recursion_limit": 20},
    context=agent_context,  # type: ignore[arg-type]
)

print("\nAgent execution complete!")
print(f"Total messages: {len(result['messages'])}")
response = result["structured_response"]

VizSelectionState prepared:
  nlp_query: Show me the monthly sales numbers for the last twelve months
  output_schema fields: 3
  materialized_data rows: 12
[values] {'messages': [HumanMessage(content='# User Request\nShow me the monthly sales numbers for the last twelve months\n\n# Data Schema\n{\n  "id": "schema_monthly_sales_v1",\n  "version": 1,\n  "fields": [\n    {\n      "name": "row_id",\n      "type": "STRING",\n      "role": "IDENTIFIER",\n      "context": "Stable identifier for each monthly record"\n    },\n    {\n      "name": "month",\n      "type": "STRING",\n      "role": "DIMENSION",\n      "context": "Month label in YYYY-MM format"\n    },\n    {\n      "name": "sales_amount",\n      "type": "FLOAT",\n      "role": "MEASURE",\n      "context": "Total sales amount in USD for the month"\n    }\n  ],\n  "primary_key": [\n    "row_id"\n  ],\n  "schema_version": 1\n}\n\n# Sample Data\n[\n  {\n    "row_id": "r1",\n    "month": "2025-01",\n    "sales_amount": 125000.0\n  },\n

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

### Try all the test cases now and compare results

In [ ]:
# Run all test cases and collect scoring stats
import json

def calculate_score(response, expected_chart_type: str) -> tuple[float, str]:
    """
    Calculate score based on chart type matching.
    
    Returns:
        (score, match_type) where match_type is 'full', 'alternative', or 'none'
    """
    # Full match = 1.0
    if response.chart_type == expected_chart_type:
        return 1.0, "full"
    
    # Check alternatives: 0.8 / number_of_alternatives
    alternatives = response.alternatives or []
    for alt in alternatives:
        if alt.chart_type == expected_chart_type:
            return 0.8 / len(alternatives), "alternative"
    
    return 0.0, "none"


# Collect results
results = []
total_score = 0.0
full_matches = 0
alt_matches = 0
misses = 0

print("=" * 80)
print("RUNNING ALL TEST CASES")
print("=" * 80)

for i, tc in enumerate(test_cases):
    test_id = tc["id"]
    expected_type = tc["expected_chart"]["chart_type"]
    
    print(f"\n{'='*80}")
    print(f"Test Case {i+1}/{len(test_cases)}: {test_id}")
    print(f"Expected chart type: {expected_type}")
    print("=" * 80)
    
    # Build the selection state
    formatted_request = format_selection_request(
        nlp_query=tc["nlp_query"],
        output_schema=tc["output_schema"],
        materialized_data=tc["materialized_data"]["rows"],
    )
    
    selection_state: VizSelectionState = {
        "messages": [HumanMessage(content=formatted_request)],
        "nlp_query": tc["nlp_query"],
        "output_schema": tc["output_schema"],
        "materialized_data": tc["materialized_data"]["rows"],
    }
    
    # Invoke the agent
    result = await selector_agent.ainvoke(
        selection_state,
        {"recursion_limit": 20},
        context=agent_context,  # type: ignore[arg-type]
    )
    
    response = result["structured_response"]
    
    # Pretty print the response
    print("\n--- Agent Response ---")
    print(response.model_dump_json(indent=2))
    
    # Calculate score
    score, match_type = calculate_score(response, expected_type)
    total_score += score
    
    if match_type == "full":
        full_matches += 1
    elif match_type == "alternative":
        alt_matches += 1
    else:
        misses += 1
    
    # Display score for this test case
    print(f"\n--- Scoring ---")
    print(f"Selected: {response.chart_type}")
    print(f"Expected: {expected_type}")
    print(f"Match type: {match_type}")
    print(f"Score: {score:.2f}")
    
    results.append({
        "test_id": test_id,
        "expected": expected_type,
        "selected": response.chart_type,
        "alternatives": [alt.chart_type for alt in (response.alternatives or [])],
        "match_type": match_type,
        "score": score,
    })

# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print(f"\nTotal test cases: {len(test_cases)}")
print(f"Full matches:     {full_matches} ({full_matches/len(test_cases)*100:.1f}%)")
print(f"Alt matches:      {alt_matches} ({alt_matches/len(test_cases)*100:.1f}%)")
print(f"Misses:           {misses} ({misses/len(test_cases)*100:.1f}%)")
print(f"\nTotal score:      {total_score:.2f} / {len(test_cases):.2f}")
print(f"Average score:    {total_score/len(test_cases):.2f}")

print("\n--- Results Table ---")
print(f"{'Test ID':<35} {'Expected':<20} {'Selected':<20} {'Match':<12} {'Score':<6}")
print("-" * 95)
for r in results:
    print(f"{r['test_id']:<35} {r['expected']:<20} {r['selected']:<20} {r['match_type']:<12} {r['score']:.2f}")